In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

# Data and analysis libraries
import polars as pl                         # Fast dataframes for financial data
import numpy as np                          # Numerical computing library
from datetime import datetime, timedelta    # Date and time operations
import random


# Machine learning libraries  
import torch                                # PyTorch framework
import torch.nn as nn                       # Neural network modules
import torch.optim as optim                 # Optimization algorithms
import research                             # Model building and training utilities


# Visualization and 
import altair as alt                        # Interactive visualization library

# data sources
import binance                              # Binance market data utilities

In [2]:
research.set_seed(42)

In [3]:
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_cols(-1)  # Show all columns

polars.config.Config

In [4]:
# Trading pair symbol
sym = 'BTCUSDT'
# time horizon of time series (time interval)
time_interval = '1d'
# Max number of auto-regressive lags
max_lags = 4
# Forecast horizon in steps 
forecast_horizon = 1
# Sharpe annualized rate (so it's independent of time frequency)
annualized_rate = research.sharpe_annualization_factor(time_interval, 365, 24)

In [5]:
import binance
print(binance.__file__)
print([x for x in dir(binance) if "download" in x.lower()])

c:\Users\micha\Documents\Code\school\cpp\classes\spring-2026\mat_4860\quant_trading_strategy\ML-Systematic-Trading-Engine\binance.py
['download_and_unzip', 'download_date_range', 'download_ohlc_timeseries', 'download_timeseries', 'download_trades']


In [6]:
# Data is already downloaded and processed into 24h OHLC CSV
# binance.download_date_range(sym, start_date, end_date)


In [7]:
ts = pl.read_csv(f"{sym}_{time_interval}_ohlc_updated.csv", try_parse_dates=True).sort('datetime')
ts


FileNotFoundError: The system cannot find the file specified. (os error 2): BTCUSDT_1d_ohlc_updated.csv

In [ ]:
research.plot_static_timeseries(ts, sym, 'close', time_interval)

In [ ]:
alt.data_transformers.enable("vegafusion")
research.plot_dyn_timeseries(ts, sym, 'close', time_interval)

### Feature Engineering

In [ ]:
price_time_series = pl.DataFrame({'price':[100.0,120.0,100.0]})
research.plot_column(price_time_series, 'price')

In [ ]:
price_time_series.with_columns(
    pl.col('price').diff().alias('delta'),
    ((pl.col('price')-pl.col('price').shift())/pl.col('price').shift()).alias('return'),
    (pl.col('price')/pl.col('price').shift()).log().alias('log_return'),
)

### Create target and lagged features

In [ ]:
ts = ts.with_columns((pl.col('close')/pl.col('close').shift(forecast_horizon)).log().alias('close_log_return'))
ts

In [ ]:
target = 'close_log_return'
lr = pl.col(target)
ts = ts.with_columns(
    lr.shift(forecast_horizon * 1).alias(f'{target}_lag_1'),
    lr.shift(forecast_horizon * 2).alias(f'{target}_lag_2'),
    lr.shift(forecast_horizon * 3).alias(f'{target}_lag_3'),
    lr.shift(forecast_horizon * 4).alias(f'{target}_lag_4'),
)
ts

In [ ]:
ts = research.add_lags(ts, target, max_lags, forecast_horizon)
ts

In [ ]:
ts = ts.drop_nulls()

In [ ]:
research.plot_distribution(ts, target, no_bins = 100)

In [ ]:
research.plot_distribution(ts, 'close', no_bins = 100)

### Build Model

In [ ]:
class LinearModel(nn.Module):
    def __init__(self, input_features):
        super(LinearModel, self).__init__()
        self.linear = nn.Linear(input_features, 1)

    def forward(self, x):
        return self.linear(x)

### Complexity of the model

In [ ]:
input_features = 1

linear_model = LinearModel(input_features)

research.print_model_info(linear_model, "Linear Model")
research.total_model_params(linear_model)

### Split by time

In [ ]:
features = ['close_log_return_lag_1']
target = 'close_log_return'
test_size = 0.25

In [ ]:
len(ts)

In [ ]:
int(len(ts) * test_size)

In [ ]:
split_idx = int(len(ts) * (1-test_size))
split_idx

In [ ]:
ts_train, ts_test = ts[:split_idx], ts[split_idx:]

In [ ]:
ts_train

In [ ]:
ts_test

In [ ]:
X_train = torch.tensor(ts_train[features].to_numpy(), dtype=torch.float32)
X_test = ts_test[features].to_torch().float()
y_train = torch.tensor(ts_train[target].to_numpy(), dtype=torch.float32)
y_test = torch.tensor(ts_test[target].to_numpy(), dtype=torch.float32)


In [ ]:
X_train

In [ ]:
X_train.shape

In [ ]:
y_train

In [ ]:
y_train.shape

In [ ]:
y_train = y_train.reshape(-1, 1)
y_train

In [ ]:
y_train.shape

In [ ]:
y_test = y_test.reshape(-1, 1)
y_test

In [ ]:
research.timeseries_train_test_split(ts, features, target, test_size)

### Batch Gradient Descent

In [ ]:
# hyperparameters
no_epochs = 1000 * 5
lr = 0.0005

# create model
model = LinearModel(len(features))
# lose function
criterion = nn.MSELoss()
# optimizer
optimizer = optim.Adam(model.parameters(), lr = lr)

print("\nTraining model...")

for epoch in range(no_epochs):
    # forward pass
    y_hat = model(X_train)
    loss = criterion(y_hat, y_train)

    # backward pass
    optimizer.zero_grad()   # 1. clear old gradients
    loss.backward()         # 2. compute new gradients
    optimizer.step()        # 3. update weights

    # check for improvement
    train_loss = loss.item()

    # logging
    if (epoch + 1) % 500 == 0:
        print(f"Epoch [{epoch+1}/{no_epochs}], Loss: {train_loss:.6f}")

print("\nLearned parameters")

for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}:\n{param.data.numpy()}")

# Evaluation
model.eval()
with torch.no_grad():
    y_hat = model(X_test)
    test_loss = criterion(y_hat, y_test)
    print(f"\nTest Loss: {test_loss.item():.6f}, Train Loss: {train_loss:.6f}")

    


### Test Trading Performance

In [ ]:
trade_results = pl.DataFrame({
    'y_hat': y_hat.squeeze(),
    'y': y_test.squeeze()
}).with_columns(
    (pl.col('y_hat').sign()==pl.col('y').sign()).alias('is_won'),
    pl.col('y_hat').sign().alias('signal'),
).with_columns(
    (pl.col('signal') * pl.col('y')).alias('trade_log_return')
).with_columns(
    pl.col('trade_log_return').cum_sum().alias('equity_curve')
)
trade_results

# note for self, trade_log_return a sign (-1 or 1) of actual y (the actual log return of actual data), because you want to see your 
# actual earnings and losings if you had acted on your signal (the magnitude of the signal not accounted for for now)

In [ ]:
research.plot_column(trade_results, 'equity_curve')

In [ ]:
trade_results = trade_results.with_columns(
    (pl.col('equity_curve')-pl.col('equity_curve').cum_max()).alias('drawdown_log')
)
trade_results

In [ ]:
max_drawdown_log = trade_results['drawdown_log'].min()
max_drawdown_log

In [ ]:
# simple drawdown (a perentage, 13% here)
drawdown_pct = np.exp(max_drawdown_log) - 1
drawdown_pct

In [ ]:
# if your equity (balance, net ownership) peak is $1000, this max drawdown means you would lose $136
equity_peak = 1000
equity_peak * drawdown_pct

In [ ]:
win_rate = trade_results['is_won'].mean()
win_rate

In [ ]:
# Expected value (win rate not end all be all, need to see how much we win or loss, take that into account in expected value). 
avg_win = trade_results.filter(pl.col('is_won')==True)['trade_log_return'].mean()
avg_loss = trade_results.filter(pl.col('is_won')==False)['trade_log_return'].mean()
ev = win_rate * avg_win + (1 - win_rate) * avg_loss
ev
# here we get a very small number, we have a very small edge

In [ ]:
total_log_return = trade_results['trade_log_return'].sum()
total_log_return
# very small number, but higher than video even tho our win rate is lower. We have a high ev. and there total log return is higher

In [ ]:
compound_return = np.exp(total_log_return)
compound_return
# very tiny, also if you factor in transaction fees, this will prb be negative

In [ ]:
# if we put 1000 dollars into this model / traded 1000 dollars with this model, we make 20 bucks. We might lose money beacuse of transaction fees
1000*compound_return

In [ ]:
# lowest peak of equity curve, important because it tells us how much money we lose if we start losing money
# its negative in this case, so if we had a thousand to begin with, at some point it will drop below 1000. If were are using leverage
# we have to be very careful of the equity trough (and drawndown as well)
equity_trough = trade_results['equity_curve'].min()
equity_trough

In [ ]:
equity_peak = trade_results['equity_curve'].max()
equity_peak
# 9% here in log space

In [ ]:
std = trade_results['trade_log_return'].std()
std
# std dev is a measure of risk (std dev of log returns)

In [ ]:
# calculate the reward to risk, using the sharpe ratio, expected value, divided by std dev, multiplied by annualized rate, which is the square root of
# the number of trading periods. This gives us our annualized sharpe. This is the most common measure of risk-adjusted returns, what's our return
# with regards to risk?

sharpe = ev / std * annualized_rate
sharpe

# this should be a high number, from 6 to double digit, but you can see here it's almost 0. Not good. This is not a good model, but its just a baseline
# we are going to learn how to benchmark our model and how we can improve on our model

In [ ]:
# all these benchmarks are quite tedious to do, so the creator has a method (in the custom research.py library) to do it all for us
# will be useful when comparing different models, and putting it al into a dataframe sorting by a certain metric, to see how the models did
research.eval_model_performance(y_test, y_hat, features, target, annualized_rate)

In [ ]:
target = 'close_log_return'
features = ['close_log_return_lag_2']
model = LinearModel(len(features))
perf = research.benchmark_reg_model(ts, features, target, model, annualized_rate, no_epochs=50)
perf

# as you can see from the weights (weight) its a mean reversion model, linear models are easier to interpret as opposed to a neutral network 
# with a billion parameters

In [ ]:
import itertools

benchmarks = []
feature_pool = [f'{target}_lag_{i}' for i in range(1, max_lags + 1)]
combos = list(itertools.combinations(feature_pool, 1))      
# itertools good for looking at combination of 2 or more features, for 1 its overkill, but good to introduce it

for features in combos:
    model = LinearModel(len(features))
    benchmarks.append(research.benchmark_reg_model(ts, list(features), target, model, annualized_rate, no_epochs=200, loss=nn.L1Loss()))
    # using minimum abs error for loss function this time, called L1Loss in PyTorch, better if your data has outliers

benchmark = pl.DataFrame(benchmarks)
benchmark.sort('sharpe', descending=True)      # sort by your preferred metric, sharpe seems like a good one

# based on these metrics lag_2 model would be the one we go with, but we haven't factored in transaction fees, let's see what happens

In [ ]:
research.auto_reg_corr_matrx(ts, target, max_lags)
# as you can see, close_log_return (target) doesn't exacly have the strongest correlation to lag_2, so dont just judge a feature by its
# correlation to the target, esp if we are customzing the learning alg and the loss function

In [ ]:
features = ['close_log_return_lag_2']
model = LinearModel(len(features))
model_trades = research.learn_model_trades(ts, features, target, model, no_epochs=200, loss=nn.L1Loss())

research.plot_column(model_trades, 'equity_curve')

# mostly positive, only some negative, more stable, which is good. But there is a big drawdown in the middle, which could be dangerous if we 
# are using a lot of leverage, like 20x. Small amout of leverage, this would suffice, so it really depends on the amount of risk you are 
# willing to tolerate. But for our purposes, we will use this to build our strategy. We will also explore if we might get liquidated if we use
# say, 8x leverage, or 10x, etc. One thing we haven't taken into account is transactions fees, we will look at that next

### Add Transaction Fees

In [ ]:
maker_fee = 0.0001      # fee for providing market liquidity (placing limits orders)
taker_fee = 0.0003      # fee for making immediate market orders 
                        # taker fee usually higher than market fee to encourage market stability, but both transactions cost 
                        # money to make the exchange momey

roundtrip_fee_log = np.log(1 - 2 * maker_fee) # two taker fees because we are open and close a trade Immediately (so market orders both times)
                                              # changed to maker fee later on to test it, usually if you want a lot of trades, the market 
                                              # rewards you for that high volume with lower fees and even rebates

model_trades = model_trades.with_columns(pl.lit(roundtrip_fee_log).alias('tx_fee_log'))
model_trades = model_trades.with_columns((pl.col("trade_log_return") + pl.col('tx_fee_log')).alias('trade_log_return_net'))
model_trades = model_trades.with_columns(pl.col('trade_log_return_net').cum_sum().alias('equity_curve_net'))
model_trades


In [ ]:
research.plot_column(model_trades, 'equity_curve_net')
# we have stable returns, but unfortunately stable negative returns. transactions fee magnifies our losses and decreases our average win,
# so its really reducing our expected value, if fees were lower maybe we'd be fine but they are high, cause us to lose money despite a >50% win
# rate and positive ev before. this is why we need to factor in transactions fees into our model. Without transaction fees, we'd have an edge
# what this tells us is that we can't trade on a 1 hour basis. With maker fees, you have lower loses but loses nonetheless. 
# Now what we need to do is to look at different time intervals, maybe longer time intervals, like 8 hours or 12 hours

In [ ]:
model_trades['is_won'].mean()   # statistics of model haven't changed, just transactions fees tanked the equity curve

In [ ]:
model_trades = research.add_tx_fees_log(model_trades, maker_fee, taker_fee)
model_trades
# to make modeling maker and taker fees easier, this add_tx_fees method has been implemented and shows the net equity for both maker and taker
# fees

In [ ]:
# Already using 8h data from the start!


In [ ]:
no_lags = 3
ts = research.add_log_return_features(ts, 'close', forecast_horizon, max_no_lags=no_lags)
ts
# we've created a new time series, and added the features, now we're gonna build the model again

In [ ]:
target = 'close_log_return'
feature_pool = [f'{target}_lag_{i}' for i in range(1, no_lags + 1)]
research.benchmark_linear_models(ts.drop_nulls(), target, feature_pool, annualized_rate, loss=nn.HuberLoss())
# new loss function huber loss (mix of mse and min abs error cause a higher sharpe)
# also a mean reversion model, quite common for interday trading to have mean reversion

In [ ]:
research.auto_reg_corr_matrx(ts.drop_nulls(), target, no_lags)
# correlation doesnt necessary mean that feature is the one to use, but here it actually looks like it is (at least
# in terms of magnitude, in terms of absolute, yeah I guess you could say its the minium correlated? cuz its negatively correlated). 
# Test empircally always

In [ ]:
research.benchmark_linear_models(ts.drop_nulls(), target, feature_pool, annualized_rate, loss=nn.MSELoss())
# mse looks exactly the same

In [ ]:
research.benchmark_linear_models(ts.drop_nulls(), target, feature_pool, annualized_rate, loss=nn.L1Loss())
# woah, sharpe is way high, because l1 handles outliers way better, doesnt penalize the as much as mse does, fantastic sharpe, double digits
# 30% compound return on investment, equity trough is postive which is really good, max drawn down is quite low, 59% winrate, higher magnitude
# losses but more wins means positive ev 

# if we had use scikit learns closed form solutions, this custom library allows us to customize the loss function easily and just be 
# experimenting we found a pretty good model, now lets plot the equity curve, take into account transaction fees, and save this model for 
# future use

In [ ]:
features = ['close_log_return_lag_1']
model = LinearModel(len(features))
model_trades = research.learn_model_trades(ts.drop_nulls(), features, target, model, loss=nn.L1Loss())
model_trades = research.add_tx_fees_log(model_trades, maker_fee, taker_fee)
research.plot_column(model_trades, 'equity_curve')
# looks almost like a straight line, with minimal drawdown, which is what you want, the higher the sharpe, the straigther the line. Sharpes of
# 20-40 is basically a straight line. This curve tells us we could use way more leverage with this. big question now, transaction fees, lets 
# see if its still amazing

In [ ]:
research.plot_column(model_trades, 'equity_curve_net_taker')    # we'll use looking at taker fees because we'll be using a taker strategy, 
                                                                # more basic than the advance maker strategies
                                                                # note to self, couldn't debug why equity curve is weird, assuming it has to do 
                                                                # with the add_tx_fees and add_tx_fees_log discrepancy in video and github, prb 
                                                                # some bug/mistake by creator 

In [ ]:
# copy and pasting this from above to plot equity curve net manually
roundtrip_fee_log = np.log(1 - 2 * taker_fee) # two taker fees because we are open and close a trade Immediately (so market orders both times)
                                              # changed to maker fee later on to test it, usually if you want a lot of trades, the market 
                                              # rewards you for that high volume with lower fees and even rebates

model_trades = model_trades.with_columns(pl.lit(roundtrip_fee_log).alias('tx_fee_log'))
model_trades = model_trades.with_columns((pl.col("trade_log_return") + pl.col('tx_fee_log')).alias('trade_log_return_net'))
model_trades = model_trades.with_columns(pl.col('trade_log_return_net').cum_sum().alias('equity_curve_net'))
model_trades
research.plot_column(model_trades, 'equity_curve_net')

# lets gooo, with fee there no negative equity, its tradable, it just amplifies the drawdowns, reduce ev, but it does resemble the gross one
# also the more we trade this, the more trading volume we have, and also the better fees we get

In [ ]:
research.plot_column(model_trades, 'equity_curve')

In [ ]:
# best practice, save models, for later use
# torch.save(model.state_dict(), '../model_weights.pth')

### Research 24h forecast horizon model, 
#### Even though the above 8h one generated a sharpe of 10.9, just to see if maybe a 24h forecast model could be it

In [ ]:
time_interval = '1d'
no_lags = 4
ts = research.add_log_return_features(ts, 'close', forecast_horizon, max_no_lags=no_lags)
ts

In [ ]:
research.benchmark_linear_models(ts.drop_nulls(), target, feature_pool, annualized_rate, max_no_features=3, loss=nn.MSELoss())
# looks promising, even mse got sharpe of 8.8, higher than 4.4 of github

In [ ]:

research.benchmark_linear_models(ts.drop_nulls(), target, feature_pool, annualized_rate, max_no_features=3, loss=nn.HuberLoss())
# lol huber loss performed exactly the same as mse, as with previous trial

In [ ]:
research.benchmark_linear_models(ts.drop_nulls(), target, feature_pool, annualized_rate, max_no_features=3, loss=nn.L1Loss(), test_size=0.25)
# yep, that 10.91 sharpe 1 feature close_log_return_lag_1 model is still the best

In [ ]:
research.benchmark_linear_models(ts.drop_nulls(), target, feature_pool, annualized_rate, max_no_features=3, loss=nn.L1Loss())
# yep, that 10.91 sharpe 1 feature close_log_return_lag_1 model is still the best

model_weight.pth is the close_log_return_lag_1 model trained by L1Loss (min abs error)